# All-in-One: RAG Pipeline on Colab

This notebook combines setup and execution in one place.

**Run all cells from top to bottom.**

---

## Part 1: Setup (Run once per session)

In [1]:
# Clone/pull code from GitHub
import os

REPO_URL = "https://github.com/Echo-Lee/RAG-embedding.git"
REPO_NAME = "RAG-embedding"

if os.path.exists(f'/content/{REPO_NAME}'):
    print(f"📥 Pulling latest changes...")
    !cd /content/{REPO_NAME} && git pull
else:
    print(f"📦 Cloning repository...")
    !git clone {REPO_URL} /content/{REPO_NAME}

os.chdir(f'/content/{REPO_NAME}')
print(f"✅ Code ready at: {os.getcwd()}")

📦 Cloning repository...
Cloning into '/content/RAG-embedding'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 61 (delta 16), reused 51 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 47.21 KiB | 3.37 MiB/s, done.
Resolving deltas: 100% (16/16), done.
✅ Code ready at: /content/RAG-embedding


In [2]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu gradio pyyaml openai tqdm

import torch
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 77.2 MB/s eta 0:00:00:00:0100:01
✅ CUDA available: True
✅ GPU: Tesla T4


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

Mounted at /content/drive
✅ Google Drive mounted


In [4]:
# Link data from Drive
!mkdir -p data/processed

DRIVE_DATA_PATH = "/content/drive/MyDrive/Capstone-Data"
!ln -sf {DRIVE_DATA_PATH}/hospital data/processed/hospital
!ln -sf {DRIVE_DATA_PATH}/corruption data/processed/corruption

print("📊 Data files:")
!ls -lh data/processed/hospital/ 2>/dev/null || echo "⚠️  Hospital data not found"
!ls -lh data/processed/corruption/ 2>/dev/null || echo "⚠️  Corruption data not found"

📊 Data files:
total 28M
-rw------- 1 root root 28M Mar  4 19:53 threads_with_summary.json
total 3.3M
-rw------- 1 root root 3.3M Mar  4 20:09 emails_group_by_thread.json


In [5]:
# Link outputs to Drive (auto-save)
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/Capstone-Outputs"
!mkdir -p {DRIVE_OUTPUT_PATH}
!rm -rf outputs && ln -s {DRIVE_OUTPUT_PATH} outputs

print(f"✅ Outputs → {DRIVE_OUTPUT_PATH}")

✅ Outputs → /content/drive/MyDrive/Capstone-Outputs


In [6]:
# Configure API keys
from google.colab import userdata
import yaml

try:
    api_key = userdata.get('AZURE_API_KEY')
    endpoint = userdata.get('AZURE_ENDPOINT')
    print("✅ Retrieved API keys from Colab Secrets")
except:
    print("⚠️  Colab Secrets not found")
    api_key = input("Enter Azure API Key: ")
    endpoint = input("Enter Azure Endpoint: ")

for dataset in ['hospital', 'corruption']:
    with open(f'experiments/{dataset}_base_template.yaml', 'r') as f:
        config = yaml.safe_load(f)
    config['azure_api_key'] = api_key
    config['azure_endpoint'] = endpoint
    with open(f'experiments/{dataset}_base.yaml', 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    print(f"✅ Created experiments/{dataset}_base.yaml")

⚠️  Colab Secrets not found
✅ Created experiments/hospital_base.yaml
✅ Created experiments/corruption_base.yaml


In [ ]:
# Add to Python path
import sys
from pathlib import Path

# Use absolute path to ensure imports work
PROJECT_ROOT = Path('/content') / REPO_NAME
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print(f"✅ Added to Python path: {PROJECT_ROOT / 'src'}")
print(f"📂 Current directory: {Path.cwd()}")
print("\n🎉 Setup complete! Now you can run the pipeline below.")

---

## Part 2: Run Pipeline

Configure and run your RAG pipeline below.

In [8]:
# Configuration
MODE = "full"          # "full" = build index, "quick" = load existing
DATASET = "hospital"   # "hospital" or "corruption"

print(f"🎯 Mode: {MODE}")
print(f"🎯 Dataset: {DATASET}")

🎯 Mode: full
🎯 Dataset: hospital


In [11]:
# Load config
from config.config import load_config

config = load_config(f'{DATASET}_base')
print(f"✅ Loaded config for {DATASET}")
print(f"   Embedding: {config.embedding_model}")
print(f"   Reranker: {config.use_reranker}")

✅ Loaded config for hospital
   Embedding: Qwen/Qwen3-Embedding-0.6B
   Reranker: True


In [ ]:
# Build or load index
if MODE == "full":
    print("🔨 Building index from scratch...")
    
    from data.loader import DataLoader
    from retrieval.indexer import FAISSIndexer
    
    loader = DataLoader(config)
    documents = loader.load()
    print(f"✅ Loaded {len(documents)} documents")
    
    indexer = FAISSIndexer(config)
    index = indexer.build_index(documents)
    print(f"✅ Built index with {index.ntotal} vectors")
    
    index_path = f"outputs/indexes/{DATASET}"
    indexer.save_index(index_path)
    print(f"✅ Index saved to {index_path} (auto-saved to Drive)")
    
    retriever_index_path = index_path
else:
    print("⚡ Quick mode: loading existing index...")
    retriever_index_path = f"outputs/indexes/{DATASET}"

In [ ]:
# Initialize retriever
from retrieval.retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker

retriever = HybridRetriever(config, index_path=retriever_index_path)
print("✅ Retriever initialized")

if config.use_reranker:
    reranker = CrossEncoderReranker(config)
    print("✅ Reranker initialized")
else:
    reranker = None

In [ ]:
# Initialize generator
from generation.rag_generator import AzureRAGGenerator

generator = AzureRAGGenerator(config)
print("✅ Generator initialized")

## Test Query (Optional)

In [ ]:
# Try a test query
test_query = "What are the main issues discussed?"  # Change this!

print(f"🔍 Query: {test_query}\n")

# Retrieve
retrieved_docs = retriever.retrieve(test_query, top_k=10, use_rerank=True)
print(f"✅ Retrieved {len(retrieved_docs)} documents\n")

# Show top 3
print("📄 Top 3 Results:")
for i, doc in enumerate(retrieved_docs[:3], 1):
    print(f"\n{i}. Score: {doc['score']:.4f}")
    print(f"   Thread: {doc['thread_id']}")
    print(f"   Content: {doc['content'][:200]}...")

# Generate answer
print("\n🤖 Generating answer...\n")
answer = generator.generate(test_query, retrieved_docs)
print(f"Answer:\n{answer}")

## Launch Gradio Demo

In [ ]:
# Launch interactive demo
from app.gradio_app import create_demo

demo = create_demo(retriever, reranker, generator, config)
demo.launch(share=True)

# Will output a public URL: https://xxxxx.gradio.live

---

## Next Steps

### To modify code:
1. Edit locally: `src/retrieval/retriever.py`, etc.
2. Commit: `git add . && git commit -m "..." && git push`
3. Update in Colab: Run first cell (git pull)
4. Restart kernel and re-run

### To reuse saved index:
- Change `MODE = "quick"` in the configuration cell
- Re-run from configuration cell onwards (skip setup)

### To switch dataset:
- Change `DATASET = "corruption"` in configuration cell
- Re-run from configuration cell onwards